In [1]:
from joblib.externals.loky import reusable_executor

from Util.Problems import Problem, solution

import math
import Util.math_functions as mathf

class P029(Problem):
    number = 29
    title = "Distinct Powers"
    description = """<p>Consider all integer combinations of $a^b$ for $2 \\le a \\le 5$ and $2 \\le b \\le 5$:
$$\\begin{array}{rrrr}
2^2=4, &2^3=8, &2^4=16, &2^5=32\\\\
3^2=9, &3^3=27, &3^4=81, &3^5=243\\\\
4^2=16, &4^3=64, &4^4=256, &4^5=1024\\\\
\\end{array}$$</p><p>If they are then placed in numerical order, with any repeats removed, we get the following sequence of $15$ distinct terms:
$$4, 8, 9, 16, 25, 27, 32, 64, 81, 125, 243, 256, 625, 1024, 3125.$$</p><p>How many distinct terms are in the sequence generated by $a^b$ for $2 \\le a \\le 100$ and $2 \\le b \\le 100$?</p>"""
    upper_limit = 100

In [2]:
p = P029()
p.describe()

## Problem 29: Distinct Powers

<p>Consider all integer combinations of $a^b$ for $2 \le a \le 5$ and $2 \le b \le 5$:
$$\begin{array}{rrrr}
2^2=4, &2^3=8, &2^4=16, &2^5=32\\
3^2=9, &3^3=27, &3^4=81, &3^5=243\\
4^2=16, &4^3=64, &4^4=256, &4^5=1024\\
\end{array}$$</p><p>If they are then placed in numerical order, with any repeats removed, we get the following sequence of $15$ distinct terms:
$$4, 8, 9, 16, 25, 27, 32, 64, 81, 125, 243, 256, 625, 1024, 3125.$$</p><p>How many distinct terms are in the sequence generated by $a^b$ for $2 \le a \le 100$ and $2 \le b \le 100$?</p>

### Solution notes
For any $a$ holds: if $a$ is prime, all values of b provide a unique result. The same goes if $a$ can not be written in the form $x^n$. When $a$ is of the form $x^n$, however, some of the values for $b$ provide results which were already discovered when $a$ was equal to $x, x^2 ...$ or $x^{n-1}$. These results can be filtered out as follows:

If $a$ can be written as $x^n$, the values it will generate, will be those of $(x^n)^b$. Therefore, if $n\times b < upper\_limit$ the values have already been calculated when calculating $x$. Furthermore, some of the values of $(x^n)^b$ will already have been calculated by $x^2, x^3 ... x^{n-1}$.

We will loop through all values for $b$ to determine if $(x^n)^b$ has been calculated before. Assuming $n < upper\_limit$, all $b < n$ have been calculated before at $(x^b)^n$. For each remaining value of $b$, if it is prime and so is $n$, this value has not been calculated before. If it is composite, loop through previous powers of x $2, 3 ... (n-1)$. If $b \times n$ is divisible by such a power, this power of $x$ has already calculated the value iff $\frac{b \times n}{previous\_power} \le upper\_limit $

In [3]:
@solution(P029, first=True, make_fast=True, warmup_args=(P029.upper_limit, ))
def counting_solutions(upper_limit):
    def get_single_factor(n):
        root = 0
        for i in range(int(math.log2(n)), 1, -1):
            integer_root = int(n ** (1/i))
            if integer_root ** i == n:
                root = integer_root
                break
        return root

    b_options = upper_limit - 1
    total_solutions = 0

    for a in range(2, upper_limit + 1):
        if mathf.is_prime(a):
            total_solutions += b_options
            continue
        single_factor = get_single_factor(a)
        if single_factor == 0:
            total_solutions += b_options
            continue
        else:
            # a can be written as x^power
            power = int(math.log(a) / math.log(single_factor))

            prime_power = mathf.is_prime(power)

            calculated_by_x = upper_limit // power

            solutions = b_options - calculated_by_x + 1

            if power > 2:
                for b in range(calculated_by_x + 1, upper_limit + 1):
                    if mathf.is_prime(b) and prime_power:
                        continue
                    for previous_power in range(2, power):
                        if (b * power) % previous_power == 0:
                            if (b * power) // previous_power <= upper_limit:
                                solutions -= 1
                                break
            total_solutions += solutions
    return total_solutions

In [4]:
p.test_once("counting_solutions")

9183 found after a separate test in 0.022100 ms by counting_solutions (first)
